# LoRA 中文问答助手 · 魔搭 ModelScope 一键版

**国内直连、无需梯子。新用户免费 100 小时 GPU 算力（24GB 显存），预装 PyTorch CUDA 环境。**

流程：装依赖 → 数据预处理 → LoRA 训练（约 1.5~3 小时）→ 效果评估。训练每轮自动存 checkpoint，中途断线重跑可续。

In [1]:
%%capture
%pip install -q transformers peft bitsandbytes accelerate datasets sentencepiece einops matplotlib modelscope

import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"          # 镜像下载模型/数据（国内快）
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| 显存", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")

In [ ]:
# ===== 1. 数据预处理 =====
import json, re, random, os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from datasets import load_dataset

DATA_REPO, SAMPLE_NUM, EVAL_NUM = "hfl/alpaca_zh_51k", 20000, 200
MAX_INS, MAX_OUT, MIN_OUT = 512, 1024, 10

def clean_example(ex):
    instruction = (ex.get("instruction") or "").strip()
    output      = (ex.get("output") or "").strip()
    input_      = (ex.get("input") or "").strip()
    if not instruction or not output:
        return None
    if len(output) < MIN_OUT:
        return None
    if len(instruction) > MAX_INS or len(output) > MAX_OUT:
        return None
    if not re.search(r"[\u4e00-\u9fffA-Za-z0-9]", output):
        return None
    return {"instruction": instruction, "input": input_, "output": output}

os.makedirs("data", exist_ok=True)
ds = load_dataset(DATA_REPO, split="train")
ds = ds.shuffle(seed=42).select(range(min(SAMPLE_NUM, len(ds))))
cleaned = [c for c in (clean_example(x) for x in ds) if c is not None]
print(f"清洗：{len(ds)} -> {len(cleaned)} 条")

ins_lens = [len(c["instruction"]) for c in cleaned]
out_lens = [len(c["output"]) for c in cleaned]
print(f"instruction 长度 均值 {np.mean(ins_lens):.1f} | output 均值 {np.mean(out_lens):.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ins_lens, bins=50, color="#4C9BE8"); axes[0].set_title("Instruction Length")
axes[1].hist(out_lens, bins=50, color="#E87A90"); axes[1].set_title("Output Length")
plt.tight_layout(); plt.savefig("data/length_distribution.png", dpi=150); plt.show()

random.Random(42).shuffle(cleaned)
train, eval_ = cleaned[: len(cleaned) - EVAL_NUM], cleaned[-EVAL_NUM:]
with open("data/train.jsonl", "w", encoding="utf-8") as f:
    for c in train: f.write(json.dumps(c, ensure_ascii=False) + "\n")
with open("data/eval.jsonl", "w", encoding="utf-8") as f:
    for c in eval_: f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"train {len(train)} 条 | eval {len(eval_)} 条 已保存")

Using the latest cached version of the dataset since hfl/alpaca_zh_51k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /root/.cache/huggingface/datasets/hfl___alpaca_zh_51k/default/0.0.0/8b1273b33cbf1f3d802c32bb00e19d33c0d7995b (last modified on Sun Sep 27 14:56:56 2026).


清洗：20000 -> 18053 条
instruction 长度 均值 18.3 | output 均值 93.3
train 17853 条 | eval 200 条 已保存


加载 Qwen2-7B-Instruct（4bit，约 15GB 需下载几分钟）...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
!git --version && git lfs version
!git lfs install
!git clone https://www.modelscope.cn/qwen/Qwen2-7B-Instruct.git models/Qwen2-7B-Instruct

git version 2.34.1
git-lfs/3.2.0 (GitHub; linux amd64; go 1.18.2)
Git LFS initialized.
正克隆到 'models/Qwen2-7B-Instruct'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 39 (delta 13), reused 39 (delta 13), pack-reused 0
接收对象中: 100% (39/39), 3.60 MiB | 14.35 MiB/s, 完成.
处理 delta 中: 100% (13/13), 完成.
过滤内容: 100% (4/4), 14.18 GiB | 16.87 MiB/s, 完成.


In [ ]:
!du -sh models/Qwen2-7B-Instruct

29G	models/Qwen2-7B-Instruct


In [2]:
# ===== 2. LoRA 训练（约 1.5~3 小时） =====
import os, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

MODEL_DIR = "models/Qwen2-7B-Instruct"
print(f"模型目录：{MODEL_DIR}")

MAX_SEQ_LEN = 1024
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
LR, EPOCHS, BATCH, GRAD_ACCUM = 2e-4, 1, 1, 16
ADAPTER_DIR = "output/lora-qwen2/final"

# fp16 计算：V100 / A10 / 3090 等各类国产平台卡均支持（bf16 仅 Ampere+ 支持，不通用）
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
print("加载 Qwen2-7B-Instruct（4bit，约 15GB 需下载几分钟）...")
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, quantization_config=bnb_config,
                                            device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

SYSTEM_PROMPT = "你是专业的中文问答助手。请直接、准确、简洁地回答问题，不要冗长铺垫。"
def format_example(ex):
    user = ex["instruction"] + ("\n\n输入：" + ex["input"] if ex.get("input", "").strip() else "")
    msgs = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user},{"role":"assistant","content":ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}
def tokenize_fn(examples):
    tok = tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

train_ds = load_dataset("json", data_files="data/train.jsonl", split="train")
train_ds = train_ds.map(format_example)
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
train_ds = train_ds.filter(lambda x: len(x["input_ids"]) > 0)
print(f"训练样本 {len(train_ds)} 条")

args = TrainingArguments(output_dir="output/lora-qwen2", per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS, learning_rate=LR,
    lr_scheduler_type="cosine", logging_steps=20,
    save_strategy="steps", save_steps=334, save_total_limit=3, fp16=True, optim="paged_adamw_8bit",
    report_to="none", remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True))
trainer.train()

os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"adapter 已保存：{ADAPTER_DIR}")

steps, losses = [], []
for log in trainer.state.log_history:
    if "loss" in log:
        steps.append(log["step"]); losses.append(log["loss"])
plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker="o", ms=3, color="#4C9BE8")
plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("LoRA Fine-tuning Loss Curve"); plt.grid(alpha=0.3)
plt.savefig("output/loss_curve.png", dpi=150); plt.show()
print(f"最终 loss：{losses[-1]:.4f}")


模型目录：models/Qwen2-7B-Instruct
加载 Qwen2-7B-Instruct（4bit，约 15GB 需下载几分钟）...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643
训练样本 17853 条


/usr/local/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,1.678940
40,1.417053
60,1.358296
80,1.348233
100,1.372608
120,1.330418
140,1.341402
160,1.313126
180,1.346522
200,1.350912


/usr/local/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

adapter 已保存：output/lora-qwen2/final
最终 loss：1.3225


In [3]:
# ===== 3. 效果评估：PPL 对比 + 同题回答对比 =====
import json, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_DIR = "models/Qwen2-7B-Instruct"
from modelscope import snapshot_download
MODEL_DIR = snapshot_download("qwen/Qwen2-7B-Instruct", local_dir=MODEL_DIR)
print(f"模型目录：{MODEL_DIR}")

ADAPTER_DIR = "output/lora-qwen2/final"
MAX_SEQ_LEN = 1024
SYSTEM_PROMPT = "你是专业的中文问答助手。请直接、准确、简洁地回答问题，不要冗长铺垫。"
QUESTIONS = ["什么是机器学习中的过拟合？", "解释一下反向传播算法的原理。",
            "Python 中列表和元组的区别是什么？", "什么是数据库索引？它有什么优缺点？",
            "如何缓解梯度消失问题？", "简单解释一下 LoRA 微调的原理。"]

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
def load_model(adapter_path=None):
    m = AutoModelForCausalLM.from_pretrained(MODEL_DIR, quantization_config=bnb_config,
                                            device_map="auto", trust_remote_code=True)
    if adapter_path:
        m = PeftModel.from_pretrained(m, adapter_path)
    m.eval(); return m

def generate(m, tok, q, max_new=200):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":q}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(p, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inp, max_new_tokens=max_new, do_sample=False)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

def compute_ppl(m, tok, texts):
    m.eval(); total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for t in texts:
            enc = {k: v.to(m.device) for k, v in tok(t, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).items()}
            out = m(**enc, labels=enc["input_ids"])
            n = (enc["input_ids"] != tok.pad_token_id).sum().item()
            total_nll += out.loss.item() * n; total_tokens += n
    return float(torch.exp(torch.tensor(total_nll / total_tokens)))

tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
texts = []
with open("data/eval.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 100: break
        ex = json.loads(line)
        user = ex["instruction"] + ("\n\n输入：" + ex["input"] if ex.get("input", "").strip() else "")
        msgs = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user},{"role":"assistant","content":ex["output"]}]
        texts.append(tok.apply_chat_template(msgs, tokenize=False))

base = load_model()
lora = load_model(ADAPTER_DIR)
ppl_base, ppl_lora = compute_ppl(base, tok, texts), compute_ppl(lora, tok, texts)
print(f"PPL 原始模型：{ppl_base:.4f} | LoRA 微调后：{ppl_lora:.4f} | 下降 {(1 - ppl_lora/ppl_base)*100:.1f}%")

lines = ["# 微调前后效果对比\n"]
for q in QUESTIONS:
    a1, a2 = generate(base, tok, q), generate(lora, tok, q)
    lines.append(f"**Q：{q}**\n\n- 原始：{a1}\n- 微调后：{a2}\n")
    print(f"\nQ：{q}\n原始：{a1[:100]}...\n微调：{a2[:100]}...")
with open("output/comparison.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("\n评估完成，产物在 output/ 目录")

2026-09-27 20:42:23,649 | INFO    | modelscope_hub.download | Downloading 15 files from qwen/Qwen2-7B-Instruct@master


Downloading:   0%|          | 0/15 [00:00<?, ?file/s]

模型目录：models/Qwen2-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

PPL 原始模型：20.1539 | LoRA 微调后：3.5211 | 下降 82.5%

Q：什么是机器学习中的过拟合？
原始：在机器学习中，过拟合（Overfitting）是指模型在训练数据上表现得过于优秀，以至于它学习到了训练数据的噪声和异常值，而非数据中的普遍规律。这种情况下，模型在训练集上的误差非常低，但在未见过的数据...
微调：过拟合是机器学习中的一种现象，当模型在训练数据上表现良好，但在测试数据上表现不佳时发生。这通常发生在模型过于复杂或过度拟合训练数据的情况下。...

Q：解释一下反向传播算法的原理。
原始：反向传播算法是一种用于训练人工神经网络的优化方法，主要应用于监督学习任务。其核心原理是通过计算损失函数关于每个权重的梯度，并利用梯度下降法来调整权重，以最小化损失函数。以下是反向传播算法的主要步骤：
...
微调：反向传播算法是一种监督学习算法，用于训练神经网络。它通过计算网络输出与期望输出之间的差异来优化网络权重。然后，该差异被反向传播回网络，以更新权重并减少误差。这个过程重复进行，直到达到所需的准确性水平。...

Q：Python 中列表和元组的区别是什么？
原始：在Python中，列表（list）和元组（tuple）都是用于存储多个元素的集合，但它们之间有以下主要区别：

1. **可变性**：
   - 列表是可变的，意味着你可以添加、删除或修改列表中的元素...
微调：列表是可变的，而元组是不可变的。这意味着列表可以添加或删除元素，而元组不能。此外，列表使用方括号，而元组使用圆括号。...

Q：什么是数据库索引？它有什么优缺点？
原始：数据库索引是一种数据结构，用于提高数据查询效率。它在表的一列或几列上创建一个额外的结构，使得能够快速定位到数据行。索引的优点包括：

1. **加速查询**：通过使用索引来定位数据，可以显著减少搜索时...
微调：数据库索引是一种数据结构，用于快速访问数据库中的记录。它通过将数据组织成有序的列表来实现这一点，使得可以快速定位和检索数据。索引的优点是它们可以显著提高查询速度，因为它们允许数据库引擎快速定位数据。然...

Q：如何缓解梯度消失问题？
原始：缓解梯度消失问题的主要策略包括：

1. **使用激活函数**：避免使用sigmoid或tanh作为激活函数，因为它们在输出接近0或1时梯度会变

In [4]:
import shutil
shutil.make_archive("lora_output", "zip", "output")
print("打包完成：lora_output.zip")

打包完成：lora_output.zip


## 训练完导出产物

在 JupyterLab 左侧文件栏找到 `output/` 目录，右键 `loss_curve.png`、`comparison.md`、`final/`（adapter，≈15MB）→ **下载**，存回本地项目即可。

本地 `4_infer_demo.py` 加载该 adapter 即可问答（无 GPU 的电脑推理较慢但可用）。